# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset loaded successfully!")
print(f"Title: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview

List all available record sets and their `@id`s. For each record set, display the available fields and their IDs for reference.

If the dataset contains multiple record sets, this helps in understanding the schema and structure.

In [ ]:
# Examine available record sets
from pprint import pprint

if hasattr(metadata, 'record_set') and metadata.record_set:
    print(f"Found {len(metadata.record_set)} record set(s):\n")
    for recset in metadata.record_set:
        recset_id = getattr(recset, '@id', 'N/A')
        recset_name = getattr(recset, 'name', 'N/A')
        print(f"RecordSet name: {recset_name}\n  @id: {recset_id}")
        if hasattr(recset, 'field'):
            print("  Fields:")
            for f in recset.field:
                # Field might be a reference (@id) or object
                if isinstance(f, str):
                    print(f"    - @id: {f}")
                elif hasattr(f, '@id'):
                    print(f"    - @id: {f.@id}, name: {getattr(f, 'name', 'N/A')}")
        print("")
else:
    print("No explicit record sets found in schema metadata.\n\nAttempting to infer from files.")
    # If no record sets: check distributions as CSV
    if hasattr(metadata, 'distribution'):
        for dist in metadata.distribution:
            dist_id = getattr(dist, '@id', 'N/A')
            url = getattr(dist, 'contentUrl', 'N/A')
            print(f"Distribution: {dist_id}\n  Content URL: {url}\n")

## 3. Data Extraction

Load records from the record set(s) into pandas DataFrame(s) for further analysis. If you identified a record set `@id` above (e.g. `'cr:main'`), use it below.

If there are multiple record sets, you can extract all into `dataframes` dict.

In [ ]:
# Attempt to gather record sets from metadata (using @id per Croissant spec)
record_set_ids = []
if hasattr(metadata, 'record_set') and metadata.record_set:
    record_set_ids = [getattr(rs, '@id', None) for rs in metadata.record_set if hasattr(rs, '@id')]

# Example fallback: if not present, attempt to guess based on common names
if not record_set_ids:
    # The dataset may expose a single (or default) record set
    # Dataset inspection via the mlcroissant library sometimes supports default extraction
    print("No explicit record_set @id found in metadata. Using 'default' to represent the records entry.")
    record_set_ids = ['default']

dataframes = {}

for record_set in record_set_ids:
    print(f"Loading records for record set @id: {record_set}")
    try:
        records = list(dataset.records(record_set=record_set))
        if records:
            dataframes[record_set] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records into DataFrame for record set {record_set}.")
            print(f"Columns: {dataframes[record_set].columns.tolist()}")
        else:
            print(f"No records yielded for {record_set}")
    except Exception as e:
        print(f"Error loading records from record set {record_set}: {e}")

# Show columns and preview data for the first available record set
if dataframes:
    primary_record_set = list(dataframes.keys())[0]
    print(f"Preview of records from record set {primary_record_set}:")
    display(dataframes[primary_record_set].head())
else:
    print("No dataframes created.")

## 4. Exploratory Data Analysis (EDA)

Apply basic data processing (filter, normalize, group) to demonstrate how to explore the dataset. **All field references should use field `@id` as schema identifier.**

Choose a numeric field from your overview (e.g., `'coverage_percent'`, `'peptide_count'`, `'molecular_weight'`).

In [ ]:
import numpy as np

# Select your record set @id and field @id (edit as appropriate)
target_record_set_id = record_set_ids[0]  # Use first available by default
df = dataframes[target_record_set_id]
# Try known numeric columns, fallback to the first numeric column if needed
possible_numeric_fields = ['coverage_percent', 'peptide_count', 'molecular_weight', 'MW', 'num_peptides', 'cr:coverage_percent', 'cr:peptide_count', 'cr:molecular_weight']

numeric_field = None
for col in df.columns:
    # Pick first numeric-style column
    if col in possible_numeric_fields or df[col].dtype.kind in 'fi':
        numeric_field = col
        break

if numeric_field is None:
    raise ValueError("No suitable numeric field available for EDA.")
print(f"Using numeric field: {numeric_field}")

# Filtering: keep rows with value > threshold
threshold = df[numeric_field].quantile(0.5) if np.issubdtype(df[numeric_field].dtype, np.number) else 0
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold:.2f}.")

# Normalization: z-score
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
print(f"Normalized '{numeric_field}' for filtered records (showing first 5 rows):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping: try group by a likely categorical field
# E.g., group by 'description', 'accession', 'sample', or first string column
group_field = None
for col in df.columns:
    if col.lower() in ['description', 'sample', 'modification', 'cr:description', 'cr:sample']:
        group_field = col
        break
if not group_field:
    # Fallback: pick first string column
    for col in df.select_dtypes(include=["object", "category"]).columns:
        group_field = col
        break

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped filtered data by '{group_field}', showing first 5 groups:")
    print(grouped_df.head())
else:
    print("No categorical field suitable for grouping found.")

## 5. Visualization

Visualize the distribution of the selected numeric field and/or its relationship to a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If grouping field is available, show average per group (top 10)
if group_field and group_field in filtered_df.columns:
    group_means = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=group_means.values, y=group_means.index)
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.xlabel(f"Mean {numeric_field}")
    plt.ylabel(group_field)
    plt.show()

## 6. Conclusion

- The FAIR² dataset was loaded and examined using the `mlcroissant` library.
- We identified and referenced data by `@id` as defined in the Croissant schema.
- Numeric columns were filtered and normalized to explore statistical properties.
- Grouping and visualization highlighted key data patterns for further study.

The notebook can be extended for domain-specific analyses, advanced visualizations, and machine learning workflows.